<a href="https://colab.research.google.com/github/vitalinamysiv/restaurant-visitor-forecasting/blob/main/notebooks/00_prepare_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Подготовка данных

## Цель

Подготовить дневной датасет для прогнозирования потока посетителей ресторана.

Исходный датасет — Rossmann Store Sales.

Итоговый датасет содержит:

date — дата наблюдения

restaurant_id — идентификатор точки

guests — количество клиентов

revenue — дневная выручка

## 1. Генерация датасета

Запускаем скрипт `src/create_dataset.py`. Он выполняет следующие шаги:

1. **Загружает** исходный `train.csv` (1 017 209 строк × 8 колонок).
2. **Переименовывает** колонки: `Store → restaurant_id`, `Customers → guests`, `Sales → revenue`, `Date → date`.
3. **Удаляет закрытые дни** (`Open == 0`) — их 172 817. Прогноз нужен только для работающих точек.
4. **Восстанавливает календарь** — добавляет пропущенные даты (например, воскресенья), чтобы получить непрерывный временной ряд.
5. **Помечает аномалии**: 54 дня, где `Sales == 0`, но магазин был открыт.
6. **Удаляет пропуски выгрузки** — 205 992 строки, где данных о продажах/клиентах нет.
7. **Добавляет календарные признаки**: `is_state_holiday`, `is_school_holiday`, `is_promo`.
8. **Сохраняет результат** в `data/processed/restaurant_daily.csv`.

Ожидаемый результат: ~844 000 строк × 7 колонок.

In [7]:
!python src/create_dataset.py

Загрузка данных...
/content/restaurant-visitor-forecasting/src/create_dataset.py:30: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
Размер исходных данных: (1017209, 8)
Закрытых дней (Open == 0): 172817
Аномалий (Sales == 0 при Open == 1): 54
Размер после восстановления календаря: (1050330, 7)
Пропусков в guests: 205992
Пропусков в revenue: 205992
Удалено строк с пропусками выгрузки: 205992
Готово!
Размер обработанных данных: (844338, 7)
Сохранено: data/processed/restaurant_daily.csv
Колонки: ['restaurant_id', 'date', 'revenue', 'guests', 'is_state_holiday', 'is_school_holiday', 'is_promo']


## 2. Первичный осмотр датасета

Загружаем подготовленный датасет и проверяем:

- **структуру** (`df_daily.head()`),
- **типы и пропуски** (`df_daily.info()`),
- **описательные статистики** (`df_daily.describe()`),
- **диапазон дат** (начало и конец периода).

Это базовый «health check» перед переходом к EDA.


In [8]:
import pandas as pd

df_daily = pd.read_csv(
    "data/processed/restaurant_daily.csv",
    parse_dates=["date"]
)

df_daily.head()

,restaurant_id,date,revenue,guests,is_state_holiday,is_school_holiday,is_promo
0,1,2013-01-02,5530.0,668.0,0,1,0
1,1,2013-01-03,4327.0,578.0,0,1,0
2,1,2013-01-04,4486.0,619.0,0,1,0
3,1,2013-01-05,4997.0,635.0,0,1,0
4,1,2013-01-07,7176.0,785.0,0,1,1


In [9]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 844338 entries, 0 to 844337
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   restaurant_id      844338 non-null  int64         
 1   date               844338 non-null  datetime64[ns]
 2   revenue            844338 non-null  float64       
 3   guests             844338 non-null  float64       
 4   is_state_holiday   844338 non-null  int64         
 5   is_school_holiday  844338 non-null  int64         
 6   is_promo           844338 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(4)
memory usage: 45.1 MB


In [10]:
df_daily.describe()

,restaurant_id,date,revenue,guests,is_state_holiday,is_school_holiday,is_promo
count,844338.000000,844338,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000
mean,558.421374,2014-04-11 01:08:38.729702656,6955.959134,762.777166,0.001078,0.193578,0.446356
min,1.000000,2013-01-01 00:00:00,46.000000,8.000000,0.000000,0.000000,0.000000
25%,280.000000,2013-08-16 00:00:00,4859.000000,519.000000,0.000000,0.000000,0.000000
50%,558.000000,2014-03-31 00:00:00,6369.000000,676.000000,0.000000,0.000000,0.000000
75%,837.000000,2014-12-11 00:00:00,8360.000000,893.000000,0.000000,0.000000,1.000000
max,1115.000000,2015-07-31 00:00:00,41551.000000,7388.000000,1.000000,1.000000,1.000000
std,321.730861,NaN,3103.815515,401.194153,0.032812,0.395102,0.497114


In [11]:
print(
    "Начало:",
    df_daily["date"].min()
)

print(
    "Конец:",
    df_daily["date"].max()
)

Начало: 2013-01-01 00:00:00
Конец: 2015-07-31 00:00:00


## 3. Выводы

По результатам подготовки данных можно зафиксировать следующие факты:

1. **Объём датасета:** 844 338 строк × 7 колонок. Пропусков нет.

2. **Период наблюдений:** с 2013-01-01 по 2015-07-31 — примерно 2.6 года истории. Этого достаточно для обучения модели с временным разделением на train/valid.

3. **Количество ресторанов:** 1115 уникальных точек. Модель различает их через признак `restaurant_id`.

4. **Целевая переменная `guests`:**
   - среднее значение ≈ 763 гостя в день;
   - медиана ≈ 676 (распределение смещено вправо);
   - минимум = 0, максимум = 7388.

5. **Выручка `revenue`:** коррелирует с `guests`, но не является целевой переменной. Используется как дополнительный признак в модели.

6. **Дополнительные признаки:**
   - `is_state_holiday` — около 0.1% дней приходится на государственные праздники;
   - `is_school_holiday` — около 19% дней приходится на школьные каникулы;
   - `is_promo` — около 45% дней приходится на промо-акции (это значимая доля, признак важен для модели).

7. **Обработанные аномалии:**
   - 172 817 закрытых дней удалено;
   - 205 992 строки с пропусками выгрузки удалено;
   - 54 аномалии (`Sales == 0` при `Open == 1`) сохранены — они могут отражать реальные события (перебои, технические сбои).

**Следующий шаг:** `notebooks/01_eda.ipynb` — разведочный анализ, сезонность и распределения.